# 3. Qwen3-VL-2B-Instruct: Baselines & Fine-Tuning

This notebook covers the experimental pipeline for Qwen3-VL-2B-Instruct on a single dataset (LaTeX_OCR only):

1. **Zero-shot and one-shot inference** — baselines before fine-tuning.
2. **Supervised Fine-Tuning (SFT) with LoRA** on LaTeX_OCR (1,164 examples, 3 epochs).
3. **Checkpoint selection** on the validation set.
4. **Results Summary** — aggregated comparison with the combined-dataset SFT from `3_qwen3vl_2datasets.ipynb`.

All results are saved to Google Drive for the comparison notebook (`4_comparison.ipynb`).

Qwen3-VL-2B-Instruct (Qwen Team, 2025) is a 2B-parameter vision-language model from the Qwen series. Its dynamic-resolution architecture (Wang et al., 2024) processes images at native aspect ratios, which is advantageous for variable-height handwritten formulas.

**Important:** Qwen3-VL requires `transformers >= 4.57.0` (installed from git). This is incompatible with SmolVLM's requirement of `transformers == 4.46.0`, which is why the two models have separate notebooks. **Do not run this notebook in the same Colab session as `2_smolvlm.ipynb` without restarting the runtime.**

**Shared modules:** This notebook imports `data_prep.py` (dataset loading) and `metrics.py` (evaluation metrics). Prompt templates and the evaluation pipeline are model-specific and defined in this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip uninstall -y huggingface-hub transformers gradio torchao
!pip install -q huggingface-hub>=1.2.0 datasets pillow sacrebleu \
    accelerate bitsandbytes peft editdistance git+https://github.com/huggingface/transformers
!pip install -q qwen_vl_utils

Found existing installation: huggingface_hub 1.27.0
Uninstalling huggingface_hub-1.27.0:
  Successfully uninstalled huggingface_hub-1.27.0
Found existing installation: transformers 5.15.0
Uninstalling transformers-5.15.0:
  Successfully uninstalled transformers-5.15.0
Found existing installation: gradio 6.24.0
Uninstalling gradio-6.24.0:
  Successfully uninstalled gradio-6.24.0
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 22.2 MB/s eta 0:00:00


In [ ]:
import gc
import json
import os
import sys
import time
from collections import Counter
from glob import glob

import datasets
from datasets import Dataset, concatenate_datasets
import editdistance
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from peft import LoraConfig, PeftModel, get_peft_model
from PIL import Image
from qwen_vl_utils import process_vision_info
import sacrebleu
import torch
from tqdm import tqdm
from transformers import (
    AutoProcessor,
    Qwen3VLForConditionalGeneration,
    Trainer,
    TrainingArguments,
)

# Ensure shared modules are importable
if os.path.exists("metrics.py"):
    sys.path.insert(0, ".")
else:
    sys.path.insert(0, "/content/drive/MyDrive/latex_ocr_project")
import data_prep
from metrics import compute_metrics, normalize_latex

In [ ]:
SEED = 42
MW_SUBSAMPLE_SIZE = 5000
MODEL_NAME = "Qwen/Qwen3-VL-2B-Instruct"
SAVE_DIR = "/content/drive/MyDrive/latex_ocr"

## 1. Data Loading

We use the data_prep module to load the LaTeX_OCR dataset (train/val/test splits). The MathWriting-human subsample loaded here is used only by `3_qwen3vl_2datasets.ipynb` for combined training. Both datasets are described in `1_eda_and_setup.ipynb` (Sections 1.1 and 1.2). The primary LaTeX_OCR dataset provides 70 test examples for evaluation.

In [ ]:
dataset = data_prep.load_latex_ocr()
mw_train_subsample = data_prep.load_mathwriting_subsample(
    subsample_size=MW_SUBSAMPLE_SIZE,
    random_state=SEED,
)

print("LaTeX_OCR splits:")
for split in dataset:
    print(f"  {split}: {len(dataset[split]):,} examples")
print(f"\nMathWriting subsample: {len(mw_train_subsample):,} examples")

README.md:   0%|          | 0.00/5.73k [00:00<?, ?B/s]

human_handwrite/train-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 16.2MB            

human_handwrite/train-00000-of-00001.par(…): downloading bytes:           |  0.00B            

human_handwrite/validation-00000-of-0000(…): reconstructing file:   0%|          |  0.00B /  961kB            

human_handwrite/validation-00000-of-0000(…): downloading bytes:           |  0.00B            

...

data/val-00000-of-00001-184984e66f80ed7a(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/229864 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7644 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/15674 [00:00<?, ? examples/s]

MathWriting subsample: 5,000 examples, columns: ['image', 'text']
LaTeX_OCR splits:
  train: 1,164 examples
  validation: 68 examples
  test: 70 examples

MathWriting subsample: 5,000 examples


## 2. Zero- and One-shot Inference

### 2.1. Prompt Template and Inference Function

In [ ]:
def predict(images, prompt_text, model, processor, device="cpu", max_new_tokens=256,
              example_text=None):
    """
    Run inference: image(s) → LaTeX string.
    images: PIL.Image or list[PIL.Image]
    prompt_text: unused (kept for API compatibility with SmolVLM pipeline).
    example_text: if provided, a two-turn one-shot conversation is constructed
                  following Brown et al. (2020). The first image is treated as
                  the demonstration example and this text as its expected output.
    """
    INSTRUCTION = "Convert this handwritten mathematical formula to LaTeX format. Output only the LaTeX code, nothing else."

    if not isinstance(images, list):
        images = [images]

    if example_text is not None and len(images) == 2:
        # One-shot: two-turn conversation (Brown et al., 2020)
        # Turn 1: user provides example image, assistant responds with ground truth
        # Turn 2: user provides query image
        messages = [
            {"role": "user", "content": [
                {"type": "image", "image": images[0].convert("RGB")},
                {"type": "text", "text": INSTRUCTION},
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": example_text}]},
            {"role": "user", "content": [
                {"type": "image", "image": images[1].convert("RGB")},
                {"type": "text", "text": INSTRUCTION},
            ]},
        ]

    else:
        # Zero-shot (or fine-tuned): single user turn with all images
        messages = [{
            "role": "user",
            "content": [
                *[{"type": "image", "image": img.convert("RGB")} for img in images],
                {"type": "text", "text": INSTRUCTION},
            ]
        }]

    # Official Qwen3-VL pattern (Qwen Team, 2025): apply_chat_template with
    # tokenize=True and return_dict=True processes images internally and
    # returns a ready-to-use tensor dict in a single call.
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = inputs.to(device)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    # Trim input tokens from generated output (Qwen Team, 2025 pattern)
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    return processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0].strip()


### 2.2. Evaluation Metrics

We evaluate using four metrics: Exact Match (raw + normalized), CER, and BLEU. All metrics are defined in `metrics.py` and imported above. See `1_eda_and_setup.ipynb` (Section 2) for definitions, explanation, and sanity check. The choice of metrics — Exact Match (raw and normalized), Character Error Rate, and BLEU — follows the best-practice evaluation framework for Image-to-LaTeX systems (Orji et al., 2023).

> **Note:** `normalize_latex()` and `compute_metrics()` are imported from `metrics.py`. The sanity check is in `1_eda_and_setup.ipynb` (Section 2).

Metrics are working as expected.

### 2.3. One-Shot Example

The one-shot example (index 0 from the training set) was selected and visualized in `1_eda_and_setup.ipynb` (Section 3). Both model notebooks use the same index to ensure comparability. Here we reload it for use in inference. The one-shot prompting approach follows the in-context learning paradigm introduced by Brown et al. (2020).

For one-shot inference, we select a single example from the training set to include in the prompt as a demonstration of the expected input-output format. The choice of example matters: it should be representative of the dataset, visually clear, and not overly complex, so that the model can learn the output format without being distracted by content difficulty. We select a formula of moderate complexity from the beginning of the training set and visually verify its quality before use.

In [ ]:
# One-shot example (selected in 1_eda_and_setup.ipynb, Section 3)
ONE_SHOT_INDEX = 0
one_shot_example = dataset["train"][ONE_SHOT_INDEX]
one_shot_image = one_shot_example["image"].convert("RGB")
one_shot_text = one_shot_example["text"]
print(f"One-shot example: {one_shot_text}")

One-shot example: z _ { 1 } = r _ { 1 } ( \cos \theta _ { 1 } + i \sin \theta _ { 1 } )


The selected one-shot example is a medium-complexity formula containing subscripts, trigonometric functions, and parentheses, making it representative of the dataset.

### 2.4. Evaluation Pipeline: Running Inference and Computing Metrics

This section defines a reusable evaluation pipeline that runs inference on all 70 test examples and computes all four metrics. Following the official Qwen3-VL inference pattern (Qwen Team, 2025), `processor.apply_chat_template` with `tokenize=True, return_dict=True` handles image processing and tokenization in a single call. The pipeline accepts the model, processor, and device as arguments, making it compatible with any prompt strategy (zero-shot, one-shot, or fine-tuned). A progress bar is printed every 10 examples for monitoring.


In [ ]:
def run_evaluation(model, processor, test_data, device="cpu",
                   max_new_tokens=256, verbose=True,
                   example_image=None, example_text=None):
    predictions = []
    references = [ex["text"] for ex in test_data]
    n = len(test_data)
    start_time = time.time()

    for i, ex in enumerate(test_data):
        if example_image is not None:
            images = [example_image.convert("RGB"), ex["image"].convert("RGB")]
        else:
            images = [ex["image"]]

        pred = predict(images, None, model, processor, device, max_new_tokens,
                      example_text=example_text)
        predictions.append(pred)

        if verbose and (i + 1) % 10 == 0:
            elapsed = time.time() - start_time
            print(f"  Processed {i+1}/{n} ({elapsed:.1f}s)")

    elapsed = time.time() - start_time
    if verbose:
        print(f"  Done. Total time: {elapsed:.1f}s ({elapsed/n:.1f}s per example)")

    return predictions, references, elapsed

### 2.5. Qwen3-VL-2B-Instruct (Qwen Team, 2025)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(MODEL_NAME)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    dtype="auto",
    device_map="auto",
)
model.eval()
print(f"Model: {MODEL_NAME}")
print(f"Device: {device}, dtype: {model.dtype}")

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.26GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

Model: Qwen/Qwen3-VL-2B-Instruct
Device: cuda, dtype: torch.bfloat16


#### 2.5.1. Zero-shot

In [ ]:
model_results = {}

print("Running zero-shot inference...")
zero_shot_preds, reference, zs_time = run_evaluation(
    model, processor, dataset["test"],
    device=device
)

Running zero-shot inference...
  Processed 10/70 (59.0s)
  Processed 20/70 (111.2s)
  Processed 30/70 (165.0s)
  Processed 40/70 (210.0s)
  Processed 50/70 (252.8s)
  Processed 60/70 (299.4s)
  Processed 70/70 (344.3s)
  Done. Total time: 344.3s (4.9s per example)


In [ ]:
zero_shot_metrics = compute_metrics(zero_shot_preds, references)
model_results["zero_shot"] = {
    "predictions": zero_shot_preds,
    "metrics": zero_shot_metrics
}

print(f"Zero-shot results:")
print(f"  Exact Match:          {zero_shot_metrics['exact_match']:.1%}")
print(f"  Exact Match (norm.):  {zero_shot_metrics['exact_match_normalized']:.1%}")
print(f"  CER:                  {zero_shot_metrics['cer']:.1%}")
print(f"  BLEU:                 {zero_shot_metrics['bleu']:.1f}")

print("\nExamples:")
for i in range(3):
    print(f"  REF: {references[i]}")
    print(f"  PRD: {zero_shot_preds[i]}")
    print()

Zero-shot results:
  Exact Match:          0.0%
  Exact Match (norm.):  0.0%
  CER:                  24.1%
  BLEU:                 73.6

Examples:
  REF: \sqrt { b ^ { 2 } - 4 a c }
  PRD: $$\sqrt{b^{2}-4ac}$$

  REF: \sqrt { x - y - z + x ^ { 2 } + y ^ { 2 } + z ^ { 2 } }
  PRD: $$\sqrt{x - y - z + x^2 + y^2 + z^2}$$

  REF: \frac { 2 \tan \alpha } { 1 - \tan ^ { 2 } \alpha }
  PRD: $$\frac{2 \tan \alpha}{1 - \tan^2 \alpha}$$



#### 2.5.2. One-shot

In [ ]:
print("Running one-shot inference...")
one_shot_preds, _, os_time = run_evaluation(
    model, processor, dataset["test"],
    device=device,
    example_image=one_shot_example["image"],
    example_text=one_shot_example["text"]
)

Running one-shot inference...
  Processed 10/70 (71.5s)
  Processed 20/70 (141.6s)
  Processed 30/70 (224.6s)
  Processed 40/70 (296.4s)
  Processed 50/70 (362.1s)
  Processed 60/70 (436.6s)
  Processed 70/70 (512.0s)
  Done. Total time: 512.0s (7.3s per example)


In [ ]:
one_shot_metrics = compute_metrics(one_shot_preds, references)
model_results["one_shot"] = {
    "predictions": one_shot_preds,
    "metrics": one_shot_metrics
}

print(f"One-shot results:")
print(f"  Exact Match:          {one_shot_metrics['exact_match']:.1%}")
print(f"  Exact Match (norm.):  {one_shot_metrics['exact_match_normalized']:.1%}")
print(f"  CER:                  {one_shot_metrics['cer']:.1%}")
print(f"  BLEU:                 {one_shot_metrics['bleu']:.1f}")

print("\nExamples:")
for i in range(3):
    print(f"  REF: {references[i]}")
    print(f"  PRD: {one_shot_preds[i]}")
    print()

One-shot results:
  Exact Match:          4.3%
  Exact Match (norm.):  10.0%
  CER:                  21.0%
  BLEU:                 81.3

Examples:
  REF: \sqrt { b ^ { 2 } - 4 a c }
  PRD: $$\sqrt { b ^ { 2 } - 4 a c }$$

  REF: \sqrt { x - y - z + x ^ { 2 } + y ^ { 2 } + z ^ { 2 } }
  PRD: $$\sqrt { x - y - z + x ^ { 2 } + y ^ { 2 } + z ^ { 2 } }$$

  REF: \frac { 2 \tan \alpha } { 1 - \tan ^ { 2 } \alpha }
  PRD: $$\frac { 2 \tan \alpha } { 1 - \tan ^ { 2 } \alpha }$$



#### 2.5.3. Exporting Results

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)

save_data = {
    "model": MODEL_NAME,
    "device": device,
    "results": {
        "zero_shot": {
            "predictions": model_results["zero_shot"]["predictions"],
            "exact_match": model_results["zero_shot"]["metrics"]["exact_match"],
            "exact_match_normalized": model_results["zero_shot"]["metrics"]["exact_match_normalized"],
            "cer": model_results["zero_shot"]["metrics"]["cer"],
            "bleu": model_results["zero_shot"]["metrics"]["bleu"],
            "inference_time_per_example": round(zs_time / len(dataset["test"]), 1),
        },
        "one_shot": {
            "predictions": model_results["one_shot"]["predictions"],
            "exact_match": model_results["one_shot"]["metrics"]["exact_match"],
            "exact_match_normalized": model_results["one_shot"]["metrics"]["exact_match_normalized"],
            "cer": model_results["one_shot"]["metrics"]["cer"],
            "bleu": model_results["one_shot"]["metrics"]["bleu"],
            "inference_time_per_example": round(os_time / len(dataset["test"]), 1),
        },
        "references": references,
        "one_shot_example": {
            "index": ONE_SHOT_INDEX,
            "text": one_shot_example["text"],
        }
    }
}

save_path = os.path.join(SAVE_DIR, "qwen3vl_zs_os_results.json")
with open(save_path, "w") as f:
    json.dump(save_data, f, ensure_ascii=False, indent=2)

print(f"Results saved to: {save_path}")

Results saved to: /content/drive/MyDrive/latex_ocr/qwen3vl_zs_os_results.json


#### 2.5.4. Results

In [ ]:
results_table = pd.DataFrame([
    {
        "Model": MODEL_NAME.split("/")[-1],
        "Setup": "Zero-shot",
        "Exact Match": f"{model_results['zero_shot']['metrics']['exact_match']:.1%}",
        "EM (norm.)": f"{model_results['zero_shot']['metrics']['exact_match_normalized']:.1%}",
        "CER": f"{model_results['zero_shot']['metrics']['cer']:.1%}",
        "BLEU": f"{model_results['zero_shot']['metrics']['bleu']:.1f}",
    },
    {
        "Model": MODEL_NAME.split("/")[-1],
        "Setup": "One-shot",
        "Exact Match": f"{model_results['one_shot']['metrics']['exact_match']:.1%}",
        "EM (norm.)": f"{model_results['one_shot']['metrics']['exact_match_normalized']:.1%}",
        "CER": f"{model_results['one_shot']['metrics']['cer']:.1%}",
        "BLEU": f"{model_results['one_shot']['metrics']['bleu']:.1f}",
    }
])

print("Evaluation Results (before SFT)")
results_table

Evaluation Results (before SFT)


,Model,Setup,Exact Match,EM (norm.),CER,BLEU
0,Qwen3-VL-2B-Instruct,Zero-shot,0.0%,0.0%,24.1%,73.6
1,Qwen3-VL-2B-Instruct,One-shot,4.3%,10.0%,21.0%,81.3


Qwen3-VL-2B-Instruct baselines are substantially stronger than SmolVLM-256M's across CER and BLEU, consistent with the 8× larger parameter count. Zero-shot CER is 24.1% (vs. 60.5% for SmolVLM), and BLEU reaches 73.6 (vs. 49.7). However, raw exact match is 0.0% and normalized EM is also 0.0% in zero-shot — despite high n-gram overlap, no single prediction matches the reference even after whitespace normalisation. This suggests systematic formatting deviations (e.g., consistent preference for certain command variants or bracket styles) rather than random errors.

Unlike SmolVLM, one-shot consistently outperforms zero-shot on all four metrics: EM 4.3% vs. 0.0%, EM (norm.) 10.0% vs. 0.0%, CER 21.0% vs. 24.1%, BLEU 81.3 vs. 73.6. The 2B model can effectively leverage the demonstration example, likely because its larger attention capacity is not disrupted by the additional image in the prompt.

In summary, Qwen3-VL's baselines are far more accurate than SmolVLM's, but still fall short of usable quality. The gap between strong CER/BLEU and zero normalized EM suggests that fine-tuning needs to correct systematic style differences rather than learn LaTeX from scratch.

## 3. Supervised Fine-Tuning

> **Dependencies.** This section relies on objects defined earlier in the notebook.
> To run Section 3 as a standalone block, ensure `dataset`, `mw_train_subsample`,
> `processor`, and all metric/helper functions from Section 2 are in scope.

### 3.1. Qwen3-VL-2B-Instruct using LoRA

This section performs SFT on Qwen3-VL-2B-Instruct using LoRA (Hu et al., 2022). The goal is to adapt the model to recognize handwritten mathematical formulas and output them in LaTeX format. The Qwen-VL architecture supports dynamic-resolution processing (Wang et al., 2024), which may provide an advantage for the variable-height formula images in our datasets. A custom data collator applies the chat template and constructs labels using `process_vision_info` for Qwen3-VL's image handling, while LoRA keeps the training memory-efficient. After training, we evaluate all epoch checkpoints on the validation set (68 examples) and select the best one for final evaluation on the test set (70 examples).

This section performs SFT on LaTeX_OCR only (1,164 examples, 3 epochs). The combined-dataset experiment (LaTeX_OCR + MathWriting-human) is in a separate notebook (`3_qwen3vl_2datasets.ipynb`) because it uses a reduced MathWriting subsample (700 vs. 5,000) and fewer epochs (2 vs. 3) to fit the free Colab T4 time limit, while keeping batch size and gradient accumulation identical.

In [ ]:
# ── Experiment selector ───────────────────────────
# Change this flag to switch between training configurations.
#   False → LaTeX_OCR only (1,164 examples)
#   True  → LaTeX_OCR + MathWriting-human (combined, 6,164 examples)
USE_COMBINED = False
# ──────────────────────────────────

In [ ]:
# Training
NUM_EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 8
LEARNING_RATE = 1e-4

In [ ]:
if USE_COMBINED:
    DATASET_NAME = "sft_2datasets"
    OUTPUT_DIR = "./qwen3vl_lora_combined"
    ADAPTER_SAVE_NAME = "qwen3vl_lora_2datasets"
    RESULTS_FILE = "qwen3vl_lora_2datasets_results.json"

    train_dataset = data_prep.get_combined_train(dataset, mw_train_subsample)
    print(f"Training on COMBINED dataset: {len(train_dataset):,} examples")
else:
    DATASET_NAME = "sft_1dataset"
    OUTPUT_DIR = "./qwen3vl_lora"
    ADAPTER_SAVE_NAME = "qwen3vl_lora_1dataset"
    RESULTS_FILE = "qwen3vl_lora_1dataset_results.json"

    train_dataset = dataset["train"]
    print(f"Training on LaTeX_OCR only: {len(train_dataset):,} examples")

train_dataset

Training on LaTeX_OCR only: 1,164 examples


Dataset({
    features: ['image', 'text'],
    num_rows: 1164
})

#### 3.1.1. LoRA Configuration and Model Loading

We use LoRA (Low-Rank Adaptation) to fine-tune only a small number of additional parameters while keeping the base model frozen. This drastically reduces VRAM usage and training time. The configuration targets all linear projection layers in the language model (q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj) with rank r=8. Rank r=8 is a commonly used value recommended by Hu et al. (2022) and has been shown to achieve a good balance between adaptation capacity and overfitting risk for parameter-efficient fine-tuning. Targeting all linear projections is the standard approach for adapting large vision-language models (Hu et al., 2022; Mangrulkar et al., 2022). The Instruct variant is used as the starting checkpoint because it already understands chat-based instructions.

In [ ]:
# Free the model from Step 2 to release GPU memory
if "model" in dir():
    del model
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(MODEL_NAME)

In [ ]:
# LoRA config targeting all linear projection layers
lora_config = LoraConfig(
    r=8,
    lora_alpha=8,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    task_type="CAUSAL_LM",
)

In [ ]:
# Load a fresh model and wrap it with LoRA adapters
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    dtype="auto",
    device_map="auto",
)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.config.use_cache = False  # Required when using gradient checkpointing with Trainer
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

trainable params: 8,716,288 || all params: 2,136,248,320 || trainable%: 0.4080


#### 3.1.2. Data Collation

A custom collator function is responsible for formatting each batch. For every training example, it constructs a two-turn conversation (user instruction with the image → assistant response with the ground-truth LaTeX) and applies the model's chat template. Unlike SmolVLM, Qwen3-VL requires images to be passed as actual PIL images in the message content (not placeholder dicts), and `process_vision_info` extracts the processed image/video tensors. Messages are constructed once and reused for both the chat template and `process_vision_info` to avoid duplication. Labels are derived from input_ids by masking padding tokens and `<|image_pad|>` tokens with -100 (which excludes them from the loss). This ensures the model learns to predict only the text tokens — both the instruction and the LaTeX response.


In [ ]:
# Image pad token for Qwen3-VL — mask it in labels so the model
# learns to predict only text tokens, not visual placeholders
image_token_id = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")

def collate_fn(examples):
    """Formats a batch of (image, text) pairs into model inputs with labels."""
    all_messages = []
    for example in examples:
        image = example["image"]
        if image.mode != "RGB":
            image = image.convert("RGB")

        messages = [
            {"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": (
                    "Convert this handwritten mathematical formula "
                    "to LaTeX format. Output only the LaTeX code, nothing else."
                )},
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": example["text"]}]}
        ]
        all_messages.append(messages)

    # Build text via chat template (single pass, no duplicate construction)
    texts = [
        processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False).strip()
        for msgs in all_messages
    ]

    # Extract processed image/video tensors via qwen_vl_utils
    batch_images = []
    batch_videos = []
    for msgs in all_messages:
        img_in, vid_in = process_vision_info(msgs)
        batch_images.extend(img_in)
        if vid_in:
            batch_videos.extend(vid_in)

    batch = processor(
        text=texts,
        images=batch_images,
        videos=batch_videos or None,
        return_tensors="pt",
        padding=True,
    )

    # Mask padding and image_pad tokens in labels (exclude from loss)
    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == image_token_id] = -100
    batch["labels"] = labels
    return batch

#### 3.1.3. Training

Training hyperparameters (epochs, batch size, learning rate, etc.) are defined in the configuration cell at the top of this section. A checkpoint is saved after each epoch (`save_strategy="epoch"`) so that we can later select the best one on the validation set, effectively implementing a post-hoc early stopping strategy (Prechelt, 1998). The learning rate of 1e-4 is the default recommended for LoRA fine-tuning in the original LoRA paper (Hu et al., 2022) and has been adopted by most VLM fine-tuning recipes, including the SmolVLM fine-tuning guide. Three epochs were chosen based on the observation that small datasets (< 10K examples) tend to overfit quickly — Orji et al. (2023) report diminishing returns after 3–5 epochs for Image-to-LaTeX tasks. Effective batch size 16 is a practical choice dictated by GPU memory constraints (T4 GPU with 16 GB VRAM) and aligns with common practice in VLM fine-tuning literature. `remove_unused_columns=False` is required because the dataset contains image and text columns that are not direct model inputs but are needed by the custom collator.

In [ ]:
total_steps = (len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)) * NUM_EPOCHS
warmup_steps = max(1, int(total_steps * 0.1))

In [ ]:
training_args = TrainingArguments(
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=warmup_steps,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    logging_steps=25,
    save_strategy="epoch",
    save_total_limit=3,
    bf16=True,
    output_dir=OUTPUT_DIR,
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    train_dataset=train_dataset,
)

trainer.train()

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
25,3.205081
50,0.592660
75,0.066695
100,0.026545
125,0.023327
150,0.017553
175,0.015055
200,0.015525


TrainOutput(global_step=219, training_loss=0.4536965858174241, metrics={'train_runtime': 14330.1881, 'train_samples_per_second': 0.244, 'train_steps_per_second': 0.015, 'total_flos': 6365411869704192.0, 'train_loss': 0.4536965858174241, 'epoch': 3.0})

#### 3.1.4. Checkpoint Selection on Validation Set (Prechelt, 1998)

Training longer does not always produce the best model, a phenomenon well-documented in the early stopping literature (Prechelt, 1998). We therefore evaluate all three epoch checkpoints on the 68-example validation set and select the best one by **normalized EM** (primary) and **CER** (tiebreaker — lower is better). Normalized EM is the standard equation-level accuracy metric for Image-to-LaTeX systems (Orji et al., 2023, §12), while CER provides a symbol-level distance measure that is more appropriate as a tiebreaker than BLEU, since CER directly penalizes structural errors (e.g., missing braces) that BLEU may overlook. The best checkpoint is then used for the final test evaluation.

In [ ]:
# Free the trained model to release GPU memory before evaluation
del model
gc.collect()
torch.cuda.empty_cache()

# Find all saved checkpoints
checkpoint_dirs = sorted(glob(f"{OUTPUT_DIR}/checkpoint-*"))
print(f"Found {len(checkpoint_dirs)} checkpoints")

# Evaluate each checkpoint on the validation set
val_results = {}

for ckpt_dir in checkpoint_dirs:
    print(f"\nEvaluating {ckpt_dir} ...")

    # Load base model + LoRA adapter from this checkpoint
    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        dtype="auto",
        device_map="auto",
    )

    model_ckpt = PeftModel.from_pretrained(base_model, ckpt_dir)
    model_ckpt.eval()

    preds, refs, _ = run_evaluation(
        model_ckpt, processor, dataset["validation"],
        device=device, verbose=False,
    )

    metrics = compute_metrics(preds, refs)
    epoch = ckpt_dir.split("-")[-1]
    val_results[epoch] = {
        "checkpoint": ckpt_dir,
        **metrics,
    }
    print(f"  EM: {metrics['exact_match']:.1%}, "
          f"EM (norm.): {metrics['exact_match_normalized']:.1%}, "
          f"CER: {metrics['cer']:.1%}, "
          f"BLEU: {metrics['bleu']:.1f}")

    # Free memory for next checkpoint
    del base_model, model_ckpt
    gc.collect()
    torch.cuda.empty_cache()

# Select best checkpoint by normalized EM, then CER as tiebreaker
best_epoch = max(val_results, key=lambda e: (val_results[e]["exact_match_normalized"],
                                              -val_results[e]["cer"]))
best_ckpt = val_results[best_epoch]["checkpoint"]
print(f"\nBest checkpoint: epoch {best_epoch} ({best_ckpt})")
print(f"  EM: {val_results[best_epoch]['exact_match']:.1%}, "
      f"EM (norm.): {val_results[best_epoch]['exact_match_normalized']:.1%}, "
      f"CER: {val_results[best_epoch]['cer']:.1%}, "
      f"BLEU: {val_results[best_epoch]['bleu']:.1f}")

Found 3 checkpoints

Evaluating ./qwen3vl_lora/checkpoint-146 ...


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

  EM: 86.8%, EM (norm.): 86.8%, CER: 1.6%, BLEU: 97.4

Evaluating ./qwen3vl_lora/checkpoint-219 ...


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

  EM: 91.2%, EM (norm.): 91.2%, CER: 1.0%, BLEU: 98.3

Evaluating ./qwen3vl_lora/checkpoint-73 ...


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

  EM: 79.4%, EM (norm.): 79.4%, CER: 5.0%, BLEU: 93.3

Best checkpoint: epoch 219 (./qwen3vl_lora/checkpoint-219)
  EM: 91.2%, EM (norm.): 91.2%, CER: 1.0%, BLEU: 98.3


#### 3.1.5. Evaluation on Test Set (70 Examples)

The selected checkpoint is evaluated on the same 70 test examples from linxy/LaTeX_OCR using the same metrics as in Section 2. The combined-dataset SFT results (from `3_qwen3vl_2datasets.ipynb`) are included in the summary table in Section 4.

In [ ]:
print(f"Loading best checkpoint: {best_ckpt} ...")

base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    dtype="auto",
    device_map="auto",
)

model_sft = PeftModel.from_pretrained(base_model, best_ckpt)
model_sft.eval()

print("Running SFT evaluation on test set (70 examples)...")
sft_preds, sft_refs, sft_time = run_evaluation(
    model_sft, processor, dataset["test"],
    device=device,
)

Loading best checkpoint: ./qwen3vl_lora/checkpoint-219 ...


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

Running SFT evaluation on test set (70 examples)...
  Processed 10/70 (59.6s)
  Processed 20/70 (114.3s)
  Processed 30/70 (172.3s)
  Processed 40/70 (229.6s)
  Processed 50/70 (281.5s)
  Processed 60/70 (340.4s)
  Processed 70/70 (395.6s)
  Done. Total time: 395.6s (5.7s per example)


In [ ]:
sft_metrics = compute_metrics(sft_preds, sft_refs)

print(f"\n{DATASET_NAME} results:")
print(f"  Exact Match:          {sft_metrics['exact_match']:.1%}")
print(f"  Exact Match (norm.):  {sft_metrics['exact_match_normalized']:.1%}")
print(f"  CER:                  {sft_metrics['cer']:.1%}")
print(f"  BLEU:                 {sft_metrics['bleu']:.1f}")

print("\nExamples:")
for i in range(5):
    print(f"  REF: {sft_refs[i]}")
    print(f"  PRD: {sft_preds[i]}")
    print()

#### 3.1.6. Exporting Results

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)

# Save LoRA adapter weights to Google Drive
adapter_save_path = os.path.join(SAVE_DIR, ADAPTER_SAVE_NAME)
model_sft.save_pretrained(adapter_save_path)
processor.save_pretrained(adapter_save_path)
print(f"LoRA adapter saved to: {adapter_save_path}")

# Save numerical results
results_this_step = {
    "predictions": sft_preds,
    "exact_match": sft_metrics["exact_match"],
    "exact_match_normalized": sft_metrics["exact_match_normalized"],
    "cer": sft_metrics["cer"],
    "bleu": sft_metrics["bleu"],
    "best_epoch": int(best_epoch),
    "validation_results": val_results,
    "inference_time_per_example": round(sft_time / len(dataset["test"]), 1),
}

save_path = os.path.join(SAVE_DIR, RESULTS_FILE)
with open(save_path, "w") as f:
    json.dump({"experiment": DATASET_NAME, "results": results_this_step}, f, ensure_ascii=False, indent=2)
print(f"Results saved to: {save_path}")

## 4. Results Summary

This section aggregates all Qwen3-VL-2B-Instruct results — zero-shot, one-shot, SFT (LaTeX_OCR only), and SFT (combined dataset, from `3_qwen3vl_2datasets.ipynb`) — into a single comparison table. Results are loaded from Google Drive.

In [ ]:
rows = []

# Zero-shot & One-shot
zs_os_path = os.path.join(SAVE_DIR, "qwen3vl_zs_os_results.json")
if os.path.exists(zs_os_path):
    with open(zs_os_path) as f:
        zs_os = json.load(f)
    for key, label in [("zero_shot", "Zero-shot"), ("one_shot", "One-shot")]:
        r = zs_os["results"][key]
        rows.append({"Setup": label,
                      "Exact Match": r['exact_match'],
                      "EM (norm.)": r['exact_match_normalized'],
                      "CER": r['cer'],
                      "BLEU": r['bleu'],
                      "Time (s/ex)": r['inference_time_per_example']})
    print(f"Loaded ZS/OS results from: {zs_os_path}")
else:
    print(f"ZS/OS results not found: {zs_os_path}")
    for label in ["Zero-shot", "One-shot"]:
        rows.append({"Setup": label, "Exact Match": None, "EM (norm.)": None,
                      "CER": None, "BLEU": None, "Time (s/ex)": None})

# SFT results
for fname, label in [("qwen3vl_lora_1dataset_results.json", "SFT (LaTeX_OCR)"),
                      ("qwen3vl_lora_2datasets_results.json", "SFT (combined)")]:
    path = os.path.join(SAVE_DIR, fname)
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        r = data["results"]
        rows.append({"Setup": label,
                      "Exact Match": r['exact_match'],
                      "EM (norm.)": r['exact_match_normalized'],
                      "CER": r['cer'],
                      "BLEU": r['bleu'],
                      "Time (s/ex)": r.get("inference_time_per_example")})
        print(f"Loaded SFT results from: {path}")
    else:
        rows.append({"Setup": label, "Exact Match": None, "EM (norm.)": None,
                      "CER": None, "BLEU": None, "Time (s/ex)": None})
        print(f"SFT results not found yet: {path}")

results_summary = pd.DataFrame(rows)

metric_cols = ["Exact Match", "EM (norm.)", "CER", "BLEU", "Time (s/ex)"]

(results_summary.style
    .background_gradient(subset=metric_cols, cmap="coolwarm", axis=0)
    .format({"Exact Match": "{:.1%}", "EM (norm.)": "{:.1%}", "CER": "{:.1%}",
             "BLEU": "{:.1f}", "Time (s/ex)": "{:.1f}"})
    .hide(axis="index")
)

Loaded ZS/OS results from: /content/drive/MyDrive/latex_ocr/qwen3vl_zs_os_results.json
Loaded SFT results from: /content/drive/MyDrive/latex_ocr/qwen3vl_lora_1dataset_results.json
Loaded SFT results from: /content/drive/MyDrive/latex_ocr/qwen3vl_lora_2datasets_results.json


Setup,Exact Match,EM (norm.),CER,BLEU,Time (s/ex)
Zero-shot,0.0%,0.0%,24.1%,73.6,4.9
One-shot,4.3%,10.0%,21.0%,81.3,7.3
SFT (LaTeX_OCR),94.3%,94.3%,0.4%,98.9,5.7
SFT (combined),85.7%,85.7%,3.3%,95.1,5.8


LoRA fine-tuning on LaTeX_OCR alone brings Qwen3-VL-2B to 94.3% exact match and 0.4% CER — the model reproduces 66 out of 70 formulas character-perfectly. BLEU reaches 98.9, indicating near-perfect n-gram overlap even on the remaining imperfect predictions. EM and EM (norm.) are identical for both SFT rows, confirming that every correct prediction is an exact character match.

Unlike SmolVLM, the combined dataset (LaTeX_OCR + 700 MathWriting examples, 2 epochs) yields lower EM than single-dataset training (85.7% vs. 94.3%). However, this drop should be interpreted with caution. The 94.3% on a single dataset may partly reflect overfitting to the LaTeX_OCR distribution — both train and test splits come from the same LaTeX_OCR source and share its background style and notation conventions, which the model can exploit during fine-tuning. The combined-dataset model trades some of this same-distribution accuracy for broader generalisation, which is likely more representative of real-world performance on photos with varied backgrounds. The trade-off is further confounded by the reduced MathWriting subsample (700 vs. 5,000 for SmolVLM) and fewer epochs (2 vs. 3).

Inference time ranges from 4.9 s/example (zero-shot) to 7.3 s/example (one-shot, which processes two images per prompt). SFT models run at 5.7–5.8 s/example, consistent with the base model's speed. These figures are 2–3× higher than SmolVLM's, reflecting the 8× larger parameter count.

## 5. Conclusion

Qwen3-VL-2B-Instruct achieves 94.3% EM (CER 0.4%) after LoRA fine-tuning on LaTeX_OCR alone — a strong result that substantially outperforms SmolVLM-256M on the same setting (70.0% EM). However, this figure should be treated as an upper bound on same-distribution performance rather than an estimate of real-world accuracy: both train and test splits come from the same LaTeX_OCR source and share its background style and notation conventions, which the model can exploit during fine-tuning.

The combined-dataset experiment (85.7% EM) trades some of this same-distribution accuracy for broader generalisation by exposing the model to formulas with varied backgrounds and notation styles. Whether this trade-off is beneficial depends on the deployment context: the single-dataset model is preferable for homogeneous inputs, while the combined model is likely more robust on diverse real-world photos. A direct comparison with SmolVLM is in `4_comparison.ipynb`.

## References

1. Orji, E.Z., Haydar, A., Erşan, İ., Mwambe, O.O. (2023). Advancing OCR Accuracy in Image-to-LaTeX Conversion—A Critical and Creative Exploration. *Applied Sciences*, 13(22), 12503. https://doi.org/10.3390/app132212503
2. Hu, E.J., Shen, Y., Wallis, P., Allen-Zhu, Z., Li, Y., Wang, S., Wang, L., Chen, W. (2022). LoRA: Low-Rank Adaptation of Large Language Models. *ICLR 2022*. https://arxiv.org/abs/2106.09685
3. Wang, P., Bai, S., Tan, S., et al. (2024). Qwen2-VL: Enhancing Vision-Language Model's Perception of the World at Any Resolution. *arXiv preprint*. https://arxiv.org/abs/2409.12191
4. Qwen Team. (2025). Qwen3-VL Technical Report. Alibaba Cloud. https://huggingface.co/Qwen/Qwen3-VL-2B-Instruct
5. Brown, T.B., Mann, B., Ryder, N., et al. (2020). Language Models are Few-Shot Learners. *NeurIPS 2020*, 33, 1877–1901. https://arxiv.org/abs/2005.14165
6. Prechelt, L. (1998). Early Stopping — But When? In *Neural Networks: Tricks of the Trade*. Springer, LNCS 1524, pp. 55–69. https://doi.org/10.1007/3-540-49430-8_3
7. Mangrulkar, S., Gugger, S., Debut, L., Belkada, Y., Paul, S. (2022). PEFT: State-of-the-art Parameter-Efficient Fine-Tuning Methods. GitHub / HuggingFace. https://github.com/huggingface/peft
8. linxy. (2024). linxy/LaTeX_OCR (Human Handwrite subset). HuggingFace Datasets. https://huggingface.co/datasets/linxy/LaTeX_OCR
9. deepcopy. (2024). deepcopy/MathWriting-human. HuggingFace Datasets. https://huggingface.co/datasets/deepcopy/MathWriting-human